# AIM 1: multiple cell lines

In [ ]:
import os
import gc
import json
import gzip
import shutil
from functools import lru_cache
from dataclasses import dataclass

import pybedtools
from pybedtools import BedTool
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from pprint import *
from scipy.stats import gaussian_kde, wasserstein_distance
from statsmodels.stats.multitest import multipletests

config

everything that changes between runs lives in this cell. the expected layout is

data/hot_loci/hot_regions_<cell>.clean.bed
data/hot_loci/merged_tf_counts_<cell>.bed
data/histone_peaks/<cell>/   (peak files for every mark, in this folder or any subfolder; the mark name must appear in the file's name or in the name of a folder above it)

the genome sizes file and the blacklist are shared by all cell lines and are looked for in data_dir, then in the notebook folder. names are matched case-insensitively, so a folder called a549 or a file called H3k27me3 still resolves. data/histone_manifests is not used

a cell line can have several peak files (experiments) for the same mark. one is selected, by peak_selection:

most_peaks: the file with the most peaks (ties go to the first file name alphabetically)
median_peaks: the file in the middle of the ranking by peak count, i.e. the most typical experiment rather than an extreme one
largest_file: the largest file on disk (only a fair comparison if the candidates are all compressed or all uncompressed)
error: stop and list the candidates, for choosing by hand

peak count is only a rough proxy for experiment quality. every candidate, its peak count and the one selected are printed and saved to results_dir/selected_peak_files.csv so the choice can be checked. to force a specific file, or if a peak file does not contain its mark in its path (for example a bare encode accession like ENCFF123ABC.bed), list it in peak_file_overrides, which takes priority over peak_selection

outputs go to results_dir/<cell line>/ (plots, per-cell-line tables) and results_dir/ (tables combining all cell lines). intermediate sorted bed files go to work_dir/<cell line>/

In [ ]:
cell_lines = ['K562']  # e.g. ['A549', 'HepG2', 'K562']

data_dir = 'data'
hot_dir = os.path.join(data_dir, 'hot_loci')
peaks_dir = os.path.join(data_dir, 'histone_peaks')
results_dir = 'results'
work_dir = 'work'

shared_files = {
    'genome': 'GRCh38_EBV.chrom.sizes.tsv',
    'blacklist': 'hg38-blacklist.v2.bed.gz',
}

cell_files = {
    'hot': 'hot_regions_{cell}.clean.bed',
    'tf_counts': 'merged_tf_counts_{cell}.bed',
}

mark_names = ['H3K4me3', 'H3K27ac', 'H3K27me3', 'H3K9me3']
peak_extensions = ('.bed', '.bed.gz', '.narrowpeak', '.narrowpeak.gz', '.broadpeak', '.broadpeak.gz')

# how to pick one file when a cell line has several for the same mark: 'most_peaks', 'median_peaks', 'largest_file' or 'error'
peak_selection = 'most_peaks'

# forces a file for one cell line and mark, e.g.
# {('A549', 'H3K4me3'): 'data/histone_peaks/A549/ENCFF123ABC.bed'}
peak_file_overrides = {}

colour_map = {
    'H3K4me3': 'red',  # active
    'H3K27ac': 'pink',  # active
    'H3K27me3': 'lightblue',  # repressive
    'H3K9me3': 'blue',  # repressive
    'bivalent': 'purple',  # both
}

bin_edges = (0, 50, 80, 90, 95, 99, 100)  # occupancy percentiles

n_perms = 200         # permutations for the mark enrichment null (100 iterations are about 6.5 minutes)
n_width_perms = 20    # shuffles for the peak width null
use_cache = True      # reuse results/<cell line>/odds_null.json if it was made with the same n_perms and peak files

memory cleanup

the analysis creates a lot of short-lived objects: every bedtools call writes a pybedtools temp file, the permutation and shuffle loops make thousands of them, and each figure stays in memory until it is closed. free_memory() closes all figures, deletes the pybedtools temp files from this session and runs the garbage collector. it is called after every stage and inside the permutation and shuffle loops

this is safe because every bed file the analysis needs later (sorted marks, clean, hot and not hot loci, binned loci, the blacklist) is saved to a named file under work_dir, and only unnamed temp files are deleted. the saved results in results_dir and the files in work_dir are never touched

In [ ]:
def free_memory():
    plt.close('all')
    pybedtools.cleanup()
    gc.collect()

find and check files

all files for all cell lines are checked up front, so every missing, misnamed or ambiguous file is reported together before any analysis starts

In [ ]:
def ci_lookup(folder, name):
    # path of `name` inside `folder`, ignoring case, or None
    if not os.path.isdir(folder):
        return None
    for f in os.listdir(folder):
        if f.lower() == name.lower():
            return os.path.join(folder, f)
    return None


def find_shared(name):
    for folder in (data_dir, '.'):
        path = ci_lookup(folder, name)
        if path:
            return path
    raise FileNotFoundError(f'{name} (looked in {data_dir}/ and the notebook folder)')


def mark_candidates(cell, mark):
    # every peak file for this mark below the cell line's folder
    if (cell, mark) in peak_file_overrides:
        path = peak_file_overrides[(cell, mark)]
        if not os.path.exists(path):
            raise FileNotFoundError(f'{path} (peak_file_overrides for {cell} {mark})')
        return [path]

    cell_dir = ci_lookup(peaks_dir, cell)
    if cell_dir is None:
        raise FileNotFoundError(f'{os.path.join(peaks_dir, cell)}/ (no histone peak folder for {cell})')

    found = []
    for root, _, names in os.walk(cell_dir):
        for f in names:
            path = os.path.join(root, f)
            if mark.lower() in os.path.relpath(path, cell_dir).lower() and f.lower().endswith(peak_extensions):
                found.append(path)
    if not found:
        raise FileNotFoundError(f'{cell_dir}/: no peak file with {mark} in its path (add it to peak_file_overrides)')
    return sorted(found)


@lru_cache(maxsize=None)
def count_peaks(path):
    opener = gzip.open if path.lower().endswith('.gz') else open
    with opener(path, 'rt') as f:
        return sum(1 for line in f if line.strip() and not line.startswith(('#', 'track', 'browser')))


def choose_mark_file(cell, mark, candidates):
    if len(candidates) == 1:
        return candidates[0]
    if peak_selection == 'error':
        raise FileNotFoundError(
            f'{cell} {mark}: {len(candidates)} files found {[os.path.basename(c) for c in candidates]} '
            f'(set peak_selection or pick one in peak_file_overrides)'
        )
    if peak_selection == 'most_peaks':
        return min(candidates, key=lambda p: (-count_peaks(p), p))
    if peak_selection == 'median_peaks':
        ranked = sorted(candidates, key=lambda p: (count_peaks(p), p))
        return ranked[(len(ranked) - 1) // 2]
    if peak_selection == 'largest_file':
        return min(candidates, key=lambda p: (-os.path.getsize(p), p))
    raise ValueError(f"peak_selection must be 'most_peaks', 'median_peaks', 'largest_file' or 'error', not {peak_selection!r}")


def resolve_files(cell):
    files, problems, report = {}, [], []
    for key, template in cell_files.items():
        name = template.format(cell=cell)
        path = ci_lookup(hot_dir, name)
        if path:
            files[key] = path
        else:
            problems.append(os.path.join(hot_dir, name))
    for mark in mark_names:
        try:
            candidates = mark_candidates(cell, mark)
            files[mark] = choose_mark_file(cell, mark, candidates)
        except FileNotFoundError as e:
            problems.append(str(e))
            continue
        for path in candidates:
            report.append({
                'cell_line': cell,
                'mark': mark,
                'file': os.path.relpath(path, peaks_dir),
                'n_peaks': count_peaks(path),
                'size_bytes': os.path.getsize(path),
                'selected': path == files[mark],
            })
    return files, problems, report


def check_data():
    problems = []
    for name in shared_files.values():
        try:
            find_shared(name)
        except FileNotFoundError as e:
            problems.append(str(e))
    for cell in cell_lines:
        problems += resolve_files(cell)[1]
    problems = list(dict.fromkeys(problems))  # the same missing folder is reported once
    if problems:
        raise FileNotFoundError('problems with data files:\n' + '\n'.join(problems))


check_data()
os.makedirs(work_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

genome = find_shared(shared_files['genome'])
blacklist_stable = BedTool(find_shared(shared_files['blacklist'])).sort(g=genome).saveas(
    os.path.join(work_dir, 'blacklist.sorted.bed')
)

peak files used

every peak file found for every mark and cell line, with the one selected for the analysis (selected = True). only marks that have more than one candidate are printed; the full table is saved to results_dir/selected_peak_files.csv

In [ ]:
peak_file_report = pd.DataFrame([row for cell in cell_lines for row in resolve_files(cell)[2]])
peak_file_report.to_csv(os.path.join(results_dir, 'selected_peak_files.csv'), index=False)

n_candidates = peak_file_report.groupby(['cell_line', 'mark'])['file'].transform('size')
if (n_candidates > 1).any():
    print(f"peak_selection = '{peak_selection}': several peak files found for some marks")
    with pd.option_context('display.width', 250, 'display.max_colwidth', 90):
        pp(peak_file_report[n_candidates > 1])
else:
    print('exactly one peak file per mark and cell line')

load one cell line

blacklist-filtered 'clean' loci (regions on the genome which exhibit *some* tf binding), split into hot (overlapping a hot region) and not hot, plus bivalent regions where the active (H3K4me3) and repressive (H3K27me3) marks overlap. loci are binned by tf occupancy percentile

if several percentile edges have the same tf count (common for integer counts in cell lines with few tfs), those bins would be empty, so the tied edges are merged and the bin is labelled with the percentile range it actually covers. cell lines with no ties get exactly the labels in bin_edges. every intermediate bed is saved to work_dir, since pybedtools temp files can be deleted by cleanup() or the os

In [ ]:
def assign_occupancy_bins(n_tfs):
    edge_values = n_tfs.quantile([p / 100 for p in bin_edges]).values

    unique_values, unique_pct = [], []
    for pct, value in zip(bin_edges, edge_values):
        if unique_values and value == unique_values[-1]:
            unique_pct[-1] = pct
        else:
            unique_values.append(value)
            unique_pct.append(pct)

    if len(unique_values) < 2:
        raise ValueError('every locus has the same tf count, so occupancy bins cannot be made')

    labels = [f'{0 if j == 1 else unique_pct[j - 1]}-{unique_pct[j]}' for j in range(1, len(unique_values))]
    return pd.cut(n_tfs, bins=unique_values, labels=labels, include_lowest=True)


@dataclass
class CellData:
    name: str
    marks: dict
    clean: BedTool
    hot: BedTool
    not_hot: BedTool
    n_hot: int
    n_not_hot: int
    clean_labeled: BedTool
    df_clean: pd.DataFrame
    binned_clean: BedTool
    bin_order: list
    peak_files: dict


def load_cell_line(cell):
    out = os.path.join(work_dir, cell)
    os.makedirs(out, exist_ok=True)
    files, _, _ = resolve_files(cell)

    marks = {
        m: BedTool(files[m]).sort(g=genome).saveas(os.path.join(out, f'{m}.sorted.bed'))
        for m in mark_names
    }
    marks['bivalent'] = (
        marks['H3K4me3'].intersect(marks['H3K27me3']).sort(g=genome).merge()
        .saveas(os.path.join(out, 'bivalent.sorted.bed'))
    )

    hot_regions = BedTool(files['hot']).sort(g=genome)
    clean = (
        BedTool(files['tf_counts']).sort(g=genome).intersect(blacklist_stable, v=True)
        .saveas(os.path.join(out, 'clean.bed'))
    )
    hot = clean.intersect(hot_regions, u=True).saveas(os.path.join(out, 'hot.bed'))
    not_hot = clean.intersect(hot_regions, v=True).saveas(os.path.join(out, 'not_hot.bed'))
    n_hot, n_not_hot = hot.count(), not_hot.count()
    assert n_hot + n_not_hot == clean.count(), 'hot + not hot should equal the clean loci'

    # one file with every clean locus tagged 1 (hot) or 0 (not hot), for the permutation counts
    hot_tagged = hot.each(lambda f: pybedtools.create_interval_from_list(list(f)[:3] + ['1'])).saveas()
    not_hot_tagged = not_hot.each(lambda f: pybedtools.create_interval_from_list(list(f)[:3] + ['0'])).saveas()
    clean_labeled = (
        BedTool.cat(hot_tagged, not_hot_tagged, postmerge=False).sort(g=genome)
        .saveas(os.path.join(out, 'clean_labeled.bed'))
    )

    df_clean = clean.to_dataframe(names=['chrom', 'start', 'end', 'tfs', 'n_tfs'])
    df_clean['occupancy_bin'] = assign_occupancy_bins(df_clean['n_tfs'])

    binned_clean = BedTool.from_dataframe(
        df_clean[['chrom', 'start', 'end', 'occupancy_bin']].astype({'occupancy_bin': str})
    ).sort(g=genome).saveas(os.path.join(out, 'binned_clean.bed'))

    return CellData(
        name=cell, marks=marks, clean=clean, hot=hot, not_hot=not_hot,
        n_hot=n_hot, n_not_hot=n_not_hot, clean_labeled=clean_labeled,
        df_clean=df_clean, binned_clean=binned_clean,
        bin_order=list(df_clean['occupancy_bin'].cat.categories),
        peak_files={m: files[m] for m in mark_names},
    )

In [ ]:
cell_data = {}
for cell in cell_lines:
    cell_data[cell] = load_cell_line(cell)
    free_memory()

for cell in cell_lines:
    os.makedirs(os.path.join(results_dir, cell), exist_ok=True)

cell_summary = pd.DataFrame([
    {
        'cell_line': cd.name,
        'n_clean': len(cd.df_clean),
        'n_hot': cd.n_hot,
        'n_not_hot': cd.n_not_hot,
        **{f'{m}_peaks': bt.count() for m, bt in cd.marks.items()},
        'occupancy_bins': ', '.join(cd.bin_order),
    }
    for cd in cell_data.values()
])
pp(cell_summary.set_index('cell_line').T)

enrichment of each mark at hot loci, odds ratio forest plot

odds ratios use the usual 2x2 table (hot / not hot against mark overlap / no overlap). if any cell of the table is zero, 0.5 is added to all four cells (haldane-anscombe) so the odds ratio and its interval stay defined

In [ ]:
def odds_from_counts(both, hot_only, mark_only, neither):
    counts = [both, hot_only, mark_only, neither]
    if 0 in counts:
        counts = [c + 0.5 for c in counts]
    both, hot_only, mark_only, neither = counts
    odds = (neither * both) / (hot_only * mark_only)
    se = np.sqrt(1 / neither + 1 / hot_only + 1 / mark_only + 1 / both)
    return odds, se


def mark_enrichment(cd):
    rows = []
    for mark_name, mark_bed in cd.marks.items():
        both = cd.hot.intersect(mark_bed, u=True).count()
        hot_only = cd.hot.intersect(mark_bed, v=True).count()
        mark_only = cd.not_hot.intersect(mark_bed, u=True).count()
        neither = cd.not_hot.intersect(mark_bed, v=True).count()

        odds, se = odds_from_counts(both, hot_only, mark_only, neither)
        z = 1.96
        rows.append({
            'cell_line': cd.name,
            'mark': mark_name,
            'odds': odds,
            'lower': np.exp(np.log(odds) - z * se),
            'upper': np.exp(np.log(odds) + z * se),
        })
    return rows


def plot_forest(enrichment):
    fig, axes = plt.subplots(1, len(cell_lines), figsize=(5 * len(cell_lines), 4),
                             sharex=True, sharey=True, squeeze=False)
    for ax, cell in zip(axes[0], cell_lines):
        sub = enrichment[enrichment['cell_line'] == cell].reset_index(drop=True)
        y_positions = np.arange(len(sub))
        xerr = [sub['odds'] - sub['lower'], sub['upper'] - sub['odds']]

        ax.errorbar(sub['odds'], y_positions, xerr=xerr, fmt='s', ms=3, capsize=6)
        ax.set_yticks(y_positions)
        ax.set_yticklabels(sub['mark'])
        ax.axvline(1, color='grey', linestyle='--')
        ax.set_xscale('log')
        ax.set_xlabel('odds ratio log scale')
        ax.set_title(cell)

    fig.suptitle('enrichment of each mark at hot loci vs rest (with 95% ci)')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'plot_mark_enrichment.png'), dpi=300)
    plt.show()


enrichment = pd.DataFrame([row for cd in cell_data.values() for row in mark_enrichment(cd)])
pp(enrichment)
plot_forest(enrichment)
free_memory()

overlap and bp-covered fraction by occupancy bin, per mark

pct_overlap is the % of loci in the bin with at least one peak of the mark (binary yes/no). it ceilings near 100% for wide, dense marks at high occupancy, so pct_bp_covered is also reported: the mean % of each locus's own length covered by the mark's peaks, which keeps changing past that ceiling. both come from one bedtools coverage call per mark

In [ ]:
def bin_metrics(cd):
    rows = []
    for mark_name, mark_bed in cd.marks.items():
        cov = cd.binned_clean.coverage(mark_bed).to_dataframe(
            names=['chrom', 'start', 'end', 'occupancy_bin',
                   'n_overlaps', 'bases_covered', 'locus_length', 'frac_covered']
        )
        cov['has_overlap'] = cov['n_overlaps'] > 0

        grouped = cov.groupby('occupancy_bin', observed=True)
        pct_overlap = grouped['has_overlap'].mean() * 100
        pct_bp_covered = grouped['frac_covered'].mean() * 100

        for b in pct_overlap.index:
            rows.append({
                'cell_line': cd.name,
                'mark': mark_name,
                'occupancy_bin': b,
                'pct_overlap': pct_overlap[b],
                'pct_bp_covered': pct_bp_covered[b],
            })
    return rows


def plot_by_bin(df, column, ylabel, title, filename):
    fig, axes = plt.subplots(1, len(cell_lines), figsize=(5 * len(cell_lines), 4),
                             sharey=True, squeeze=False)
    for ax, cell in zip(axes[0], cell_lines):
        bin_order = cell_data[cell].bin_order
        for mark_name in cell_data[cell].marks:
            sub = df[(df['cell_line'] == cell) & (df['mark'] == mark_name)].set_index('occupancy_bin').reindex(bin_order)
            ax.plot(bin_order, sub[column], marker='o', label=mark_name, color=colour_map[mark_name])
        ax.set_xlabel('occupancy percentile bin')
        ax.set_yscale('log')
        ax.set_title(cell)
    axes[0][0].set_ylabel(ylabel)
    axes[0][-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, filename), dpi=300, bbox_inches='tight')
    plt.show()


bins_df = pd.DataFrame([row for cd in cell_data.values() for row in bin_metrics(cd)])
pp(bins_df)

plot_by_bin(bins_df, 'pct_overlap', '% of loci overlapping mark',
            'binary overlap by occupancy bin, per mark', 'plot_overlap_by_bin.png')
plot_by_bin(bins_df, 'pct_bp_covered', 'mean % of locus length covered by mark',
            'bp-covered fraction by occupancy bin, per mark', 'plot_bp_covered_by_bin.png')
free_memory()

permutation test for the odds ratios

the observed odds ratios are formalised by shuffling each mark's bed file n_perms times (same chromosome, blacklist excluded), recomputing the odds ratio each time, and taking the fraction of null odds ratios at least as extreme as the observed one. the bivalent null re-shuffles H3K4me3 and H3K27me3 separately and rebuilds the bivalent regions from the shuffles. p values are corrected for the number of marks tested with benjamini-hochberg fdr, within each cell line

null odds ratios are cached per cell line, so re-running the notebook does not repeat the shuffling unless n_perms or the selected peak files change (set use_cache = False to force it)

In [ ]:
def odds_for_mark(cd, mark_bed):
    hits = cd.clean_labeled.intersect(mark_bed, c=True)  # one bedtools call
    df = hits.to_dataframe(names=['chrom', 'start', 'end', 'is_hot', 'overlap_count'])

    both = int(((df['is_hot'] == 1) & (df['overlap_count'] > 0)).sum())
    mark_only = int(((df['is_hot'] == 0) & (df['overlap_count'] > 0)).sum())
    odds, _ = odds_from_counts(both, cd.n_hot - both, mark_only, cd.n_not_hot - mark_only)
    return float(odds)


def shuffle_peaks(bt):
    return bt.shuffle(g=genome, excl=blacklist_stable.fn, chrom=True).saveas()


def permute_odds(cd, n):
    null = {mark_name: [] for mark_name in cd.marks}
    for mark_name, mark_bed in cd.marks.items():
        for i in range(n):
            if mark_name == 'bivalent':
                shuffled_k4 = shuffle_peaks(cd.marks['H3K4me3'])
                shuffled_k27 = shuffle_peaks(cd.marks['H3K27me3'])
                shuffled = shuffled_k27.intersect(shuffled_k4).sort(g=genome).merge().saveas()
            else:
                shuffled = shuffle_peaks(mark_bed)

            null[mark_name].append(odds_for_mark(cd, shuffled))
            pybedtools.cleanup()  # this iteration's shuffles and bedtools outputs
    return null


def get_odds_null(cd):
    path = os.path.join(results_dir, cd.name, 'odds_null.json')
    signature = {m: [p, os.path.getsize(p)] for m, p in cd.peak_files.items()}
    if use_cache and os.path.exists(path):
        cached = json.load(open(path))
        if cached['n_perms'] == n_perms and cached.get('peak_files') == signature:
            print(f'{cd.name}: using cached null from {path}')
            return cached['odds_null']
    print(f'{cd.name}: running {n_perms} permutations per mark')
    null = permute_odds(cd, n_perms)
    json.dump({'n_perms': n_perms, 'peak_files': signature, 'odds_null': null}, open(path, 'w'))
    return null


def permutation_table(cell, enrichment, odds_null):
    rows = []
    sub = enrichment[enrichment['cell_line'] == cell]
    for mark_name, observed_odds in zip(sub['mark'], sub['odds']):
        odds_array = np.array(odds_null[mark_name])

        if observed_odds >= 1:
            p = (np.sum(odds_array >= observed_odds) + 1) / (len(odds_array) + 1)
        else:
            p = (np.sum(odds_array <= observed_odds) + 1) / (len(odds_array) + 1)

        rows.append({
            'cell_line': cell,
            'mark': mark_name,
            'observed_or': observed_odds,
            'null_median': np.median(odds_array),
            'null_p2.5': np.percentile(odds_array, 2.5),
            'null_p97.5': np.percentile(odds_array, 97.5),
            'p_value': p,
        })

    table = pd.DataFrame(rows)
    table['p_adjusted'] = multipletests(table['p_value'], method='fdr_bh')[1]
    return table


def plot_null_boxplots(odds_null_by_cell, enrichment):
    fig, axes = plt.subplots(1, len(cell_lines), figsize=(7 * len(cell_lines), 5),
                             sharey=True, squeeze=False)
    for ax, cell in zip(axes[0], cell_lines):
        odds_null = odds_null_by_cell[cell]
        names = list(odds_null.keys())
        bp = ax.boxplot([np.array(odds_null[m]) for m in names], tick_labels=names,
                        showfliers=False, patch_artist=True)
        for patch in bp['boxes']:
            patch.set(facecolor='lightgrey', edgecolor='grey')

        # overlay each mark's observed OR as a single red point
        observed = enrichment[enrichment['cell_line'] == cell].set_index('mark')['odds']
        for i, mark_name in enumerate(names):
            ax.scatter(i + 1, observed[mark_name], color='red', zorder=3, s=60,
                       label='observed OR' if i == 0 else None)

        ax.set_yscale('log')
        ax.set_title(f'{cell} ({len(odds_null[names[0]])} iterations)')
        ax.axhline(1, color='black', linestyle=':', linewidth=0.8)
        ax.tick_params(axis='x', rotation=20)
    axes[0][0].set_ylabel('odds ratio (log scale)')
    axes[0][-1].legend(loc='upper right', fontsize=9)
    fig.suptitle('null distribution vs observed odds, by mark')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'plot_permutation_boxplot.png'), dpi=300)
    plt.show()


odds_null_by_cell = {cell: get_odds_null(cd) for cell, cd in cell_data.items()}

perm_results = pd.concat(
    [permutation_table(cell, enrichment, odds_null_by_cell[cell]) for cell in cell_lines],
    ignore_index=True
)
pp(perm_results)
plot_null_boxplots(odds_null_by_cell, enrichment)
free_memory()

does the artifact scale with mark width? width-vs-null-OR diagnostic

if the null OR for each mark is driven mainly by how wide or numerous its peaks are (rather than anything positional), it should scale with total peak width. a mark sitting close to the trend line supports 'this mark's null enrichment is explained by width alone'; a mark well below it is not explained by width and needs a different (biological) explanation

caveat: with only 5 marks per cell line this is a coarse, illustrative diagnostic, not a well-powered regression - treat the trend line as a visual aid, not a statistical test

In [ ]:
def mark_width_stats(cd, odds_null):
    rows = []
    for mark_name, mark_bed in cd.marks.items():
        merged_mark = mark_bed.sort(g=genome).merge()
        n_peaks = merged_mark.count()
        total_bp = sum(len(iv) for iv in merged_mark)
        rows.append({
            'cell_line': cd.name,
            'mark': mark_name,
            'n_peaks': n_peaks,
            'total_bp': total_bp,
            'mean_width': total_bp / n_peaks if n_peaks else np.nan,
            'null_median_or': np.median(odds_null[mark_name]),
        })
    return rows


def plot_width_confound(mark_stats):
    fig, axes = plt.subplots(1, len(cell_lines), figsize=(6 * len(cell_lines), 5), squeeze=False)
    for ax, cell in zip(axes[0], cell_lines):
        sub = mark_stats[mark_stats['cell_line'] == cell]
        x, y = sub['total_bp'], sub['null_median_or']

        ax.scatter(x, y, s=60, zorder=3)
        for _, row in sub.iterrows():
            ax.annotate(row['mark'], (row['total_bp'], row['null_median_or']),
                        textcoords='offset points', xytext=(6, 4), fontsize=9)

        slope, intercept = np.polyfit(np.log(x), np.log(y), 1)
        x_line = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, np.exp(intercept) * x_line ** slope, color='grey', linestyle='--',
                label=f'trend (slope={slope:.2f})')

        ax.set_xscale('log')
        ax.set_yscale('log')
        ax.set_xlabel('total merged peak width (bp, log scale)')
        ax.set_ylabel('null-median odds ratio (log scale)')
        ax.set_title(cell)
        ax.legend()
    fig.suptitle('width-confound diagnostic: null OR vs total peak width')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'plot_width_confound.png'), dpi=300)
    plt.show()


mark_stats = pd.DataFrame([
    row for cell, cd in cell_data.items() for row in mark_width_stats(cd, odds_null_by_cell[cell])
])
pp(mark_stats)
plot_width_confound(mark_stats)
free_memory()

distribution of peak widths across occupancy bins

if a mark's association with high-occupancy loci reflects the same technical bias as the tf signal, the widths of its peaks at those loci should differ from its widths at ordinary tf-bound loci. but width also shifts for purely geometric reasons: a peak of width w overlaps a locus of length L with probability proportional to (w + L), so wider peaks are over-sampled, and most strongly at short loci. hot loci are longer than other loci, so some change in width across bins is expected by chance alone

each observed distribution is therefore compared against a width-aware null: the binned loci (keeping their bin labels and lengths) are shuffled at random n_width_perms times and the widths of the peaks that overlap them are recomputed

for each mark, every peak overlapping at least one locus in a bin contributes its width to that bin. a peak overlapping several loci in the same bin is counted once, and a peak spanning loci in several bins is counted in each of them. the bivalent 'peaks' are the merged k4me3 and k27me3 intersections, so their widths are the widths of those merged regions

In [ ]:
def peak_widths_by_bin(cd, loci_bt):
    out = []
    for mark_name, mark_bed in cd.marks.items():
        hits = mark_bed.cut([0, 1, 2]).intersect(loci_bt, wa=True, wb=True)
        if hits.count() == 0:
            continue
        df = hits.to_dataframe(
            names=['chrom', 'start', 'end', 'l_chrom', 'l_start', 'l_end', 'occupancy_bin']
        )
        df = df.drop_duplicates(['chrom', 'start', 'end', 'occupancy_bin'])
        df['mark'] = pd.Categorical([mark_name] * len(df), categories=list(cd.marks))
        df['occupancy_bin'] = pd.Categorical(df['occupancy_bin'], categories=cd.bin_order)
        df['width'] = df['end'] - df['start']
        out.append(df[['mark', 'occupancy_bin', 'width']])
    return pd.concat(out, ignore_index=True)


def width_null(cd):
    nulls = []
    for i in range(n_width_perms):
        shuffled_loci = cd.binned_clean.shuffle(g=genome, excl=blacklist_stable.fn, chrom=True, seed=i).saveas()
        df = peak_widths_by_bin(cd, shuffled_loci)
        df['perm'] = i
        nulls.append(df)
        pybedtools.cleanup()  # this shuffle and the intersect outputs
    return pd.concat(nulls, ignore_index=True)


def w1(a, b):
    return wasserstein_distance(a, b) if len(a) and len(b) else np.nan


def median_bp(a):
    return 10 ** np.median(a) if len(a) else np.nan


def noise_floor(perms):
    if len(perms) < 2:
        return np.nan
    return np.mean([w1(p, np.concatenate(perms[:i] + perms[i + 1:])) for i, p in enumerate(perms)])


def width_analysis(cd):
    obs = peak_widths_by_bin(cd, cd.binned_clean)
    obs['log_width'] = np.log10(obs['width'])
    null = width_null(cd)
    null['log_width'] = np.log10(null['width'])
    null = null.drop(columns='width')

    none = np.array([])
    obs_w = {k: g['log_width'].values for k, g in obs.groupby(['mark', 'occupancy_bin'], observed=True)}
    null_w = {k: g['log_width'].values for k, g in null.groupby(['mark', 'occupancy_bin'], observed=True)}
    null_perms = {
        k: [p['log_width'].values for _, p in g.groupby('perm')]
        for k, g in null.groupby(['mark', 'occupancy_bin'], observed=True)
    }

    summary = []
    for mark_name in cd.marks:
        for b in cd.bin_order:
            k = (mark_name, b)
            o, nl = obs_w.get(k, none), null_w.get(k, none)
            summary.append({
                'cell_line': cd.name,
                'mark': mark_name,
                'occupancy_bin': b,
                'n_peaks': len(o),
                'median_bp': median_bp(o),
                'null_median_bp': median_bp(nl),
                'w1_vs_null': w1(o, nl),
                'w1_noise': noise_floor(null_perms.get(k, [])),
            })

    # only the pooled null per mark is kept (for the ecdf); the full shuffled table is the largest object in the
    # notebook and is freed when this function returns
    null_pooled = {m: g['log_width'].values for m, g in null.groupby('mark', observed=True)}

    return {'obs': obs, 'obs_w': obs_w, 'null_pooled': null_pooled, 'summary': pd.DataFrame(summary)}


widths = {}
for cell, cd in cell_data.items():
    widths[cell] = width_analysis(cd)
    free_memory()
width_summary = pd.concat([w['summary'] for w in widths.values()], ignore_index=True)

with pd.option_context('display.width', 200, 'display.max_columns', None):
    pp(width_summary.round(3))

the table above gives, for each mark and bin, the median width of overlapping peaks next to the shuffled-loci null. w1_vs_null is the wasserstein distance between the observed and null log10 widths: the average gap between matching quantiles, so 0.1 is roughly a 26% change in width. w1_noise is the same distance between a single null permutation and the remaining permutations, i.e. what sampling noise alone produces at this sample size. w1_vs_null needs to clearly exceed w1_noise before it means anything

first plot: median width of overlapping peaks by occupancy bin, one panel per cell line. dashed lines are the shuffled-loci null for the same mark, so the gap between a solid and dashed line is the part of the width that geometry does not explain

In [ ]:
fig, axes = plt.subplots(1, len(cell_lines), figsize=(5 * len(cell_lines), 4), sharey=True, squeeze=False)
for ax, cell in zip(axes[0], cell_lines):
    bin_order = cell_data[cell].bin_order
    for mark_name in cell_data[cell].marks:
        sub = width_summary[
            (width_summary['cell_line'] == cell) & (width_summary['mark'] == mark_name)
        ].set_index('occupancy_bin').reindex(bin_order)
        ax.plot(bin_order, sub['median_bp'], marker='o', label=mark_name, color=colour_map[mark_name])
        ax.plot(bin_order, sub['null_median_bp'], linestyle='--', color=colour_map[mark_name])
    ax.set_xlabel('occupancy percentile bin')
    ax.set_yscale('log')
    ax.set_title(cell)
axes[0][0].set_ylabel('median width of overlapping peaks (bp)')
axes[0][-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
fig.suptitle('peak width by occupancy bin, per mark (dashed = shuffled-loci null)')
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'plot_peak_width_by_bin.png'), dpi=300, bbox_inches='tight')
plt.show()

full distributions, one figure per cell line and one panel per mark (lowest occupancy bin at the bottom of the ridgeline, highest at the top)

ecdf of log10 peak width, one line per occupancy bin. the black dashed line is the null pooled across bins, so it is a reference rather than a bin-matched comparison (the bin-matched version is w1_vs_null above). parallel shifted curves mean the whole distribution moved; curves that cross mean the shape changed, and a left tail (narrow peaks) that grows with occupancy is the point-source signature

histograms compare the lowest and highest occupancy bins, to look for a mixture. an ecdf hides modes, whereas a second narrow population that grows with occupancy would pull the median down without moving either component

ridgeline: one ridge per occupancy bin, a kernel density estimate of the observed widths with the median marked by the white tick. ridges are scaled to a common height within each panel (not to equal area), and bins with fewer than 5 peaks are left blank because a density estimate from so few points is meaningless. n is shown beside each bin label

In [ ]:
none = np.array([])


def mark_grid(n_marks, ncols=3, **kwargs):
    # one panel per mark plus one spare panel for the legend
    nrows = int(np.ceil((n_marks + 1) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 4 * nrows), **kwargs)
    axes = axes.ravel()
    for ax in axes[n_marks + 1:]:
        ax.set_visible(False)
    axes[n_marks].axis('off')
    return fig, axes[:n_marks], axes[n_marks]


def finish_grid(fig, axes, cell, title, xlabel, ylabel, filename):
    for ax in axes:
        ax.set_xlabel(xlabel)
        ax.tick_params(labelbottom=True)
    for ax in axes[::3]:
        ax.set_ylabel(ylabel)
    fig.suptitle(f'{cell}: {title}')
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, cell, filename), dpi=300)
    plt.show()


def ecdf(x):
    x = np.sort(x)
    return x, np.arange(1, len(x) + 1) / len(x)


def plot_width_ecdf(cell):
    cd, w = cell_data[cell], widths[cell]
    bin_colours = plt.cm.viridis(np.linspace(0, 1, len(cd.bin_order)))
    fig, axes, legend_ax = mark_grid(len(cd.marks), sharex=True, sharey=True)

    for ax, mark_name in zip(axes, cd.marks):
        for colour, b in zip(bin_colours, cd.bin_order):
            x = w['obs_w'].get((mark_name, b), none)
            if len(x):
                ax.plot(*ecdf(x), color=colour, label=b)
        pooled_null = w['null_pooled'].get(mark_name, none)
        if len(pooled_null):
            ax.plot(*ecdf(pooled_null), color='black', linestyle='--', label='shuffled null (pooled)')
        ax.set_title(mark_name)

    handles, labels = axes[0].get_legend_handles_labels()
    legend_ax.legend(handles, labels, title='occupancy bin', loc='center')
    finish_grid(fig, axes, cell, 'ecdf of peak width', 'log10 peak width (bp)', 'ecdf', 'plot_peak_width_ecdf.png')


def plot_width_hist(cell):
    cd, w = cell_data[cell], widths[cell]
    lo_bin, hi_bin = cd.bin_order[0], cd.bin_order[-1]
    hist_edges = np.linspace(w['obs']['log_width'].min(), w['obs']['log_width'].max(), 40)
    fig, axes, legend_ax = mark_grid(len(cd.marks), sharex=True)

    for ax, mark_name in zip(axes, cd.marks):
        for b, colour in ((lo_bin, 'grey'), (hi_bin, 'red')):
            x = w['obs_w'].get((mark_name, b), none)
            if len(x):
                ax.hist(x, bins=hist_edges, density=True, histtype='step', color=colour, label=b)
        ax.set_title(mark_name)

    handles, labels = axes[0].get_legend_handles_labels()
    legend_ax.legend(handles, labels, title='occupancy bin', loc='center')
    finish_grid(fig, axes, cell, 'lowest vs highest occupancy bin', 'log10 peak width (bp)', 'density',
                'plot_peak_width_hist.png')


def plot_width_ridgeline(cell, overlap=1.8):
    # overlap: how far the tallest ridge reaches into the one above it
    cd, w = cell_data[cell], widths[cell]
    median_outline = [pe.withStroke(linewidth=3, foreground='black')]
    x_grid = np.linspace(w['obs']['log_width'].min(), w['obs']['log_width'].max(), 300)
    bin_colours = plt.cm.viridis(np.linspace(0, 1, len(cd.bin_order)))

    def ridge_density(x):
        if len(x) < 5 or np.std(x) == 0:
            return None
        return gaussian_kde(x)(x_grid)

    fig, axes, legend_ax = mark_grid(len(cd.marks), sharex=True)

    for ax, mark_name in zip(axes, cd.marks):
        dens = {b: ridge_density(w['obs_w'].get((mark_name, b), none)) for b in cd.bin_order}
        available = [d.max() for d in dens.values() if d is not None]
        if not available:
            ax.set_title(mark_name)
            continue
        scale = overlap / max(available)

        for i, (b, colour) in enumerate(zip(cd.bin_order, bin_colours)):
            z = 10 * (len(cd.bin_order) - i)  # lower ridges drawn in front of higher ones
            if dens[b] is not None:
                top = i + dens[b] * scale
                ax.fill_between(x_grid, i, top, color=colour, alpha=0.85, zorder=z)
                ax.plot(x_grid, top, color='black', linewidth=0.8, zorder=z + 1)

                med = np.median(w['obs_w'][(mark_name, b)])
                ax.plot([med, med], [i, i + np.interp(med, x_grid, dens[b]) * scale],
                        color='white', linewidth=1.5, zorder=z + 2, path_effects=median_outline)

        ax.set_yticks(range(len(cd.bin_order)))
        ax.set_yticklabels([f"{b} (n={len(w['obs_w'].get((mark_name, b), none))})" for b in cd.bin_order], fontsize=8)
        ax.set_title(mark_name)

    legend_ax.legend(
        handles=[
            Patch(facecolor='grey', edgecolor='black', label='observed (kde)'),
            Line2D([0], [0], color='white', linewidth=1.5, label='observed median', path_effects=median_outline),
        ],
        loc='center'
    )
    finish_grid(fig, axes, cell, 'peak width ridgeline', 'log10 peak width (bp)', 'occupancy bin',
                'plot_peak_width_ridgeline.png')


for cell in cell_lines:
    plot_width_ecdf(cell)
    plot_width_hist(cell)
    plot_width_ridgeline(cell)
    free_memory()

save tables

tables combining every cell line go to results_dir (one row per cell line, mark and bin, with a cell_line column), and each cell line also gets its own copy under results_dir/<cell line>/

In [ ]:
tables = {
    'cell_summary': cell_summary,
    'enrichment': enrichment,
    'bin_metrics': bins_df,
    'permutation_results': perm_results,
    'mark_width_stats': mark_stats,
    'peak_width_summary': width_summary,
}

for name, table in tables.items():
    table.to_csv(os.path.join(results_dir, f'{name}.csv'), index=False)
    for cell in cell_lines:
        table[table['cell_line'] == cell].to_csv(os.path.join(results_dir, cell, f'{name}.csv'), index=False)

print(sorted(os.listdir(results_dir)))

final cleanup

frees figures, pybedtools temp files and python garbage one last time. the results in results_dir are kept. the intermediate beds in work_dir are also kept by default, since they make a re-run quicker; uncomment the last line to delete them (they are rebuilt from data_dir on the next run)

pybedtools.cleanup(remove_all=True) is deliberately not used: it deletes temp files from every pybedtools session, including other notebooks that are running at the same time

In [ ]:
free_memory()

# shutil.rmtree(work_dir)